# 03. Indicadores turísticos de Petén

**Observatorio de Tikal y Flores · Osman de León**

Este notebook calcula y revisa indicadores sobre los CSV generados por `src.etl`. Las unidades y limitaciones se conservan explícitamente. No modifica las fuentes originales.

## 1. Preparación
Ejecutar desde el repositorio. Seleccionar el kernel del entorno virtual. Primero deben existir los nueve CSV del ETL.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.indicadores import cargar, indicadores, ejecutar
DATA = ROOT / "data" / "processed"
print("Proyecto:", ROOT)

Proyecto: c:\Users\Emadlg\Desktop\UVG\proyecto3_Data


## 2. Cargar y calcular
Los cálculos se implementan en `src/indicadores.py` para evitar duplicar lógica en notebooks y dashboard.

In [2]:
tablas = cargar(DATA)
resultados = indicadores(tablas)
for nombre, tabla in resultados.items():
    print(f"{nombre}: {len(tabla)} filas")

visitantes_subdestino: 30 filas
estadia_ponderada: 30 filas
segmentacion_visitantes: 30 filas
tikal_indicadores: 116 filas
aimm_pasajeros_indicadores: 248 filas
aimm_operaciones_indicadores: 104 filas
ocupacion_indicadores: 36 filas
capacidad_hotelera: 15 filas
comparacion_fuentes_indices: 6 filas


## 3. Visitantes por subdestino
Son conteos por polígono y mes. **No sumar entre subdestinos**: una persona puede aparecer en más de uno.

In [3]:
vis = resultados["visitantes_subdestino"]
display(vis.head())
fig = px.line(vis, x="fecha", y="visitantes", color="subdestino", markers=True, title="Visitantes por polígono (no total Petén)")
fig.show()

,fecha,subdestino,visitantes,dias_persona,alcance
0,2025-09-01,El Remate Y Alrededores Lago,80968.948584,196001.988470,Polígono individual; no sumar entre subdestinos
1,2025-09-01,Isla De Flores,90024.761099,264456.609523,Polígono individual; no sumar entre subdestinos
2,2025-09-01,Mundo Maya Aeropuerto,99370.640787,250624.034537,Polígono individual; no sumar entre subdestinos
3,2025-10-01,El Remate Y Alrededores Lago,91322.522482,218636.003827,Polígono individual; no sumar entre subdestinos
4,2025-10-01,Isla De Flores,94173.304500,272718.169956,Polígono individual; no sumar entre subdestinos


## 4. Estadía media ponderada
Solo para el segmento **mayor a un día**: suma de días-persona / suma de visitantes. No promediar promedios individuales.

In [4]:
est = resultados["estadia_ponderada"]
display(est.head())
px.line(est, x="fecha", y="estadia_media_ponderada_dias", color="subdestino", markers=True, title="Estadía media ponderada (días)").show()

,fecha,subdestino,visitantes,dias_persona,estadia_media_ponderada_dias,alcance
0,2025-09-01,El Remate Y Alrededores Lago,28469.221902,143502.261789,5.040611,Solo visitantes con pernocta (>1 día)
1,2025-09-01,Isla De Flores,34594.799822,209026.648247,6.042141,Solo visitantes con pernocta (>1 día)
2,2025-09-01,Mundo Maya Aeropuerto,36358.438642,187611.832392,5.160063,Solo visitantes con pernocta (>1 día)
3,2025-10-01,El Remate Y Alrededores Lago,30808.712989,158122.194333,5.132386,Solo visitantes con pernocta (>1 día)
4,2025-10-01,Isla De Flores,36040.106153,214584.971609,5.954060,Solo visitantes con pernocta (>1 día)


## 5. Turistas frente a excursionistas
La proporción se calcula dentro de cada subdestino y mes. No se suman polígonos.

In [5]:
seg = resultados["segmentacion_visitantes"]
display(seg.head())
px.line(seg, x="fecha", y="proporcion_turistas", color="subdestino", markers=True, title="Proporción con pernocta").show()

,fecha,subdestino,turistas_pernocta,excursionistas,visitantes_poligono,proporcion_turistas,proporcion_excursionistas
0,2025-09-01,El Remate Y Alrededores Lago,28469.221902,52499.726681,80968.948584,0.351607,0.648393
1,2025-09-01,Isla De Flores,34594.799822,55429.961276,90024.761099,0.384281,0.615719
2,2025-09-01,Mundo Maya Aeropuerto,36358.438642,63012.202145,99370.640787,0.365887,0.634113
3,2025-10-01,El Remate Y Alrededores Lago,30808.712989,60513.809494,91322.522482,0.337362,0.662638
4,2025-10-01,Isla De Flores,36040.106153,58133.198347,94173.304500,0.382700,0.617300


## 6. Tikal
Las categorías nacionales, extranjeros y otros corresponden a boletos. Los meses con cero se señalan para revisión; no se asumen cierres confirmados.

In [6]:
tik = resultados["tikal_indicadores"]
print("Meses con cero:", tik.loc[tik.cero_por_confirmar,"fecha"].dt.strftime("%Y-%m").tolist())
px.line(tik, x="fecha", y="total", markers=True, title="Boletos registrados en Tikal").show()

Meses con cero: ['2017-01', '2020-04', '2020-05', '2020-06', '2020-07', '2020-08', '2020-09']


## 7. Aeropuerto AIMM
Pasajeros y operaciones se muestran en gráficas separadas. **No son visitantes únicos ni turistas necesariamente**.

In [7]:
air = resultados["aimm_pasajeros_indicadores"]
ops = resultados["aimm_operaciones_indicadores"]
px.line(air.groupby(["fecha","flujo"],as_index=False).pasajeros.sum(),x="fecha",y="pasajeros",color="flujo",title="Movimientos de pasajeros por flujo").show()
px.line(ops,x="fecha",y="operaciones",color="tipo_operacion",title="Operaciones aéreas").show()

## 8. Hotelería
La ocupación es una tasa mensual. La oferta corresponde a inventarios por corte: no se deben sumar trimestres para estimar capacidad anual.

In [8]:
ocup = resultados["ocupacion_indicadores"]
oferta = resultados["capacidad_hotelera"]
px.line(ocup,x="fecha",y="ocupacion_pct",markers=True,title="Ocupación hotelera (%)").show()
display(oferta.head(10))

,anio,trimestre,tipo_alojamiento,establecimientos,habitaciones,plazas_cama,tipo_dato
0,2022,4,Hospedaje,157,2345,6325,inventario al corte; no acumular trimestres
1,2022,4,Hotel,83,1540,4668,inventario al corte; no acumular trimestres
2,2022,4,Pensión,16,219,684,inventario al corte; no acumular trimestres
3,2023,4,Hospedaje,145,2180,5987,inventario al corte; no acumular trimestres
4,2023,4,Hotel,75,1488,4518,inventario al corte; no acumular trimestres
5,2023,4,Pensión,15,192,587,inventario al corte; no acumular trimestres
6,2024,4,Hospedaje,145,2070,5744,inventario al corte; no acumular trimestres
7,2024,4,Hotel,74,1519,4559,inventario al corte; no acumular trimestres
8,2024,4,Pensión,18,250,754,inventario al corte; no acumular trimestres
9,2025,4,Hospedaje,154,2102,5986,inventario al corte; no acumular trimestres


## 9. Triangulación preliminar
Los tres universos son diferentes. Para mostrar tendencias comparables sin confundir unidades se crean índices relativos (primer mes disponible = 100). No implica correlación causal ni equivalencia de visitantes.

In [9]:
cmp = resultados["comparacion_fuentes_indices"]
display(cmp.head())
cols = [c for c in cmp if c.startswith("indice_")]
long = cmp.melt(id_vars="fecha",value_vars=cols,var_name="fuente",value_name="indice")
px.line(long,x="fecha",y="indice",color="fuente",markers=True,title="Tendencias relativas de las tres fuentes (base 100)").show()

,fecha,boletos_tikal,pasajeros_aimm,movilidad_poligono_aeropuerto,indice_boletos_tikal,indice_pasajeros_aimm,indice_movilidad_poligono_aeropuerto,nota
0,2025-09-01,17758,27377,99370.640787,100.000000,100.000000,100.000000,Series con universos distintos; índices relati...
1,2025-10-01,23408,31931,104281.761180,131.816646,116.634401,104.942225,Series con universos distintos; índices relati...
2,2025-11-01,41411,38603,112807.669063,233.196306,141.005223,113.522131,Series con universos distintos; índices relati...
3,2025-12-01,53209,37335,137352.271183,299.633968,136.373598,138.222185,Series con universos distintos; índices relati...
4,2026-03-01,45946,38450,127506.123433,258.734092,140.446360,128.313677,Series con universos distintos; índices relati...


## 10. Guardar resultados y conclusiones
El script guarda los indicadores en `data/processed/indicadores/` sin sobrescribir los CSV base. Revisar cobertura, ceros y unidades antes de interpretar resultados.

In [10]:
salidas = ejecutar(DATA)
print("Indicadores guardados en:", DATA / "indicadores")

visitantes_subdestino.csv: 30 filas
estadia_ponderada.csv: 30 filas
segmentacion_visitantes.csv: 30 filas
tikal_indicadores.csv: 116 filas
aimm_pasajeros_indicadores.csv: 248 filas
aimm_operaciones_indicadores.csv: 104 filas
ocupacion_indicadores.csv: 36 filas
capacidad_hotelera.csv: 15 filas
comparacion_fuentes_indices.csv: 6 filas
Indicadores guardados en: c:\Users\Emadlg\Desktop\UVG\proyecto3_Data\data\processed\indicadores


### Limitaciones y siguientes pasos
- La telefonía mide presencia/visitantes estimados en polígonos solapados.
- El aeropuerto registra movimientos de pasajeros y operaciones, no turistas únicos.
- Los boletos de Tikal registran entradas y sus ceros necesitan contexto.
- Los indicadores no sustituyen el pronóstico ene–feb 2026 ni el análisis de nacionalidades, que requerirá preservar el desglose original.
- El siguiente notebook será `04_predicciones.ipynb`.